In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import sys
sys.path.append('/content/Nepali-Decoder')

In [41]:
import ijson
import src.paths as paths
import importlib

importlib.reload(src.paths)

print(paths.CHUNKS)
print(paths.RAW)

print(paths.__file__)

/content/drive/MyDrive/Nepali-Decoder/Chunked_Data_Raw
/content/drive/MyDrive/Nepali-Decoder/Raw_Data/nepali-books.json
/content/Nepali-Decoder/src/paths.py


In [42]:
def open_new_chunk(chunk_idx, out_dir):
    outfile = open(f'{out_dir}/chunk_{chunk_idx:04d}.txt', 'w', encoding='utf-8')
    return outfile


def flush_buffer(buffer, outfile):
    if not buffer:
        return
    outfile.write('\n'.join(buffer))
    outfile.write('\n')
    buffer.clear()

In [43]:
def stream_extract_chunks(filepath, out_dir):

    doc_count = 0
    chunk_idx = 0
    buffer = []
    outfile = None

    BUFFER_SIZE = 10          # flush after this many docs OR immediately if a doc is huge
    LARGE_TEXT_CHARS = 200_000  # flush immediately if a single text is this big
    TARGET_SIZE = 100 * 1024 * 1024  # 100 mb per chunk file

    with open(filepath, 'rb') as inline:
        parser = ijson.parse(inline)

        current_doc_text = None   # this doc's text_data, once we hit it

        for prefix, event, value in parser:

            if prefix == 'item' and event == 'start_map':
                # new document starting
                current_doc_text = None

            elif event == 'string' and prefix.endswith('.text_data'):
                # only the value we actually want ever touches memory here
                current_doc_text = value

            elif prefix == 'item' and event == 'end_map':
                # document finished — same behavior as the original per-doc loop body
                if outfile is None:
                    outfile = open_new_chunk(chunk_idx=chunk_idx, out_dir=out_dir)

                if current_doc_text:
                    buffer.append(current_doc_text.strip())
                    if len(buffer) >= BUFFER_SIZE or len(current_doc_text) >= LARGE_TEXT_CHARS:
                        flush_buffer(buffer, outfile)

                if outfile.tell() >= TARGET_SIZE:
                    flush_buffer(buffer, outfile)
                    outfile.close()
                    chunk_idx += 1
                    outfile = open_new_chunk(chunk_idx=chunk_idx, out_dir=out_dir)

                doc_count += 1
                if doc_count % 1000 == 0:
                    print(f"{doc_count} documents chunked")

    if buffer:
        flush_buffer(buffer, outfile)
    if outfile:
        outfile.close()


In [44]:
stream_extract_chunks(
    filepath=paths.RAW,
    out_dir=paths.CHUNKS,
)

1000 documents chunked
2000 documents chunked
3000 documents chunked
4000 documents chunked
5000 documents chunked
6000 documents chunked
7000 documents chunked


In [51]:
import unicodedata

INPUT_DIR = paths.CHUNKS

OUTPUT_DIR = paths.PROJECT_ROOT / "Normalized_Data"

OUTPUT_DIR.mkdir(exist_ok=True)

count = 0

for file in sorted(INPUT_DIR.glob('*.txt')):
    print(f'Processing file_no: {file.name}')

    with open(file, 'r', encoding='utf-8') as infile, \
        open(OUTPUT_DIR / file.name, 'w', encoding='utf-8') as outfile:

        for line in infile:
            line = unicodedata.normalize('NFC', line)
            outfile.write(line)

print("Done")


Processing file_no: chunk_0000.txt
Processing file_no: chunk_0001.txt
Processing file_no: chunk_0002.txt
Processing file_no: chunk_0003.txt
Processing file_no: chunk_0004.txt
Processing file_no: chunk_0005.txt
Processing file_no: chunk_0006.txt
Processing file_no: chunk_0007.txt
Processing file_no: chunk_0008.txt
Processing file_no: chunk_0009.txt
Processing file_no: chunk_0010.txt
Processing file_no: chunk_0011.txt
Processing file_no: chunk_0012.txt
Processing file_no: chunk_0013.txt
Processing file_no: chunk_0014.txt
Processing file_no: chunk_0015.txt
Processing file_no: chunk_0016.txt
Processing file_no: chunk_0017.txt
Processing file_no: chunk_0018.txt
Processing file_no: chunk_0019.txt
Processing file_no: chunk_0020.txt
Processing file_no: chunk_0021.txt
Processing file_no: chunk_0022.txt
Processing file_no: chunk_0023.txt
Processing file_no: chunk_0024.txt
Processing file_no: chunk_0025.txt
Processing file_no: chunk_0026.txt
Processing file_no: chunk_0027.txt
Processing file_no: 

In [ ]:
folder = paths.PROJECT_ROOT / "Normalized_Data"

normalized_files = 0
not_normalized = 0

for file in sorted(folder.glob("*.txt")):

    with open(file, encoding="utf-8") as f:
        text = f.read()

    if unicodedata.is_normalized("NFC", text):
        normalized_files += 1
    else:
        not_normalized += 1
        print(file.name, "NOT normalized")

print()
print(f"Normalized: {normalized_files}")
print(f"Not normalized: {not_normalized}")